# Work IQ A2A

The [Work IQ MCP notebook](workiq-mcp.ipynb) calls tools. This optional notebook instead delegates a task to Work IQ as a remote agent through **A2A (Agent-to-Agent)**. Discover its Agent Card, then send a natural-language question over JSON-RPC.

> In the Skillable lab, Work IQ authentication is already configured.

## Before you run the notebook

**Note**
- When prompted to select a kernel, select **Select Kernel > Python Environments** and choose **ignite26-prel42** with the path **.venv\Scripts\python.exe**.
- If prompted with "Do you want to allow public and private networks to access this app?", select **Allow**.

**Troubleshooting:** If code cells get stuck and keep spinning, first confirm that the selected kernel is the lab's **.venv\Scripts\python.exe**. Then try these steps in order:
1. Select **Restart** from the notebook toolbar.
2. If that doesn't work, select **Developer: Reload Window** from the VS Code command palette.
3. If that doesn't work, close VS Code completely and restart it. Reopen the lab folder and notebook. If cells still get stuck, let a proctor know.

After each recovery step, reselect the lab kernel if needed and rerun the setup and sign-in cells before continuing.

## Step 1: Authenticate as the signed-in user

A2A uses the same delegated Work IQ scope as MCP. Load the app registration from `.env` and sign in with your lab account in the browser when prompted.

In [ ]:
import json
import os

import requests
from dotenv import load_dotenv
from msal import PublicClientApplication

load_dotenv(override=True)

app_id = os.environ["WORK_IQ_SEARCH_ENTRA_APP_ID"]
tenant_id = os.environ["WORK_IQ_SEARCH_ENTRA_TENANT_ID"]

app = PublicClientApplication(
    client_id=app_id,
    authority=f"https://login.microsoftonline.com/{tenant_id}",
)
auth_result = app.acquire_token_interactive(["api://workiq.svc.cloud.microsoft/WorkIQAgent.Ask"])
if "access_token" not in auth_result:
    raise RuntimeError(f"Authentication failed: {auth_result.get('error_description', auth_result)}")

token = auth_result["access_token"]
print("Signed in to Work IQ")

## Step 2: Discover the Agent Card

An Agent Card describes the agent's identity, endpoint, capabilities, authentication, and skills. Inspect the card to see which tasks Work IQ advertises.

In [ ]:
response = requests.get(
    "https://workiq.svc.cloud.microsoft/a2a/.well-known/agent-card.json",
    headers={"Authorization": f"Bearer {token}"},
    timeout=60,
)
response.raise_for_status()
card = response.json()
print(json.dumps(card, indent=2))

## Step 3: Delegate an email-summary task

Send a JSON-RPC `message/send` request containing a user message with text parts and a unique message ID. Request a JSON response rather than live streaming. Work IQ retrieves the signed-in user's context and returns a task with artifacts or a message.

In [ ]:
from uuid import uuid4

rpc = {
    "jsonrpc": "2.0",
    "id": str(uuid4()),
    "method": "message/send",
    "params": {
        "message": {
            "kind": "message",
            "role": "user",
            "messageId": str(uuid4()),
            "parts": [{
                "kind": "text",
                "text": "Summarize my recent emails about the Professional Claw Hammer and their requested actions.",
            }],
        },
    },
}
response = requests.post(
    "https://workiq.svc.cloud.microsoft/a2a/",
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    },
    json=rpc,
    timeout=180,
)
response.raise_for_status()
reply = response.json()

if "error" in reply:
    raise RuntimeError(f"Work IQ A2A request failed: {reply['error']}")
result = reply["result"]

parts = list(result.get("parts", []))
for artifact in result.get("artifacts", []):
    parts.extend(artifact.get("parts", []))

answers = []
for part in parts:
    if part.get("kind") == "text":
        answers.append(part["text"])

if result.get("status", {}).get("state") in {"failed", "canceled", "rejected"} or not answers:
    raise RuntimeError(f"Work IQ A2A returned no successful answer: {result}")
print("\n\n".join(answers))

## ✅ Mission complete

**What you built:**

* ✓ **Agent Card discovery**: Work IQ's published capabilities, read from its Agent Card.
* ✓ **Delegated task**: A JSON-RPC `message/send` request that Work IQ answered with your email context.

MCP exposes tools for your code or agent to call; A2A lets your agent delegate a task to a remote peer. Both Work IQ paths use delegated authentication and the signed-in user's permissions.

➡️ Return to lab instructions.